In [ ]:
#import library dan variabel global

import os
import random
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from skimage.feature import hog, local_binary_pattern
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = "data"
IMG_SIZE = 128 # semua citra di-resize ke 128x128
TEST_SIZE = 0.2 # data test 20% dari total data
N_AUG = 2 # jumlah citra augmentasi per citra train
OUTPUT_PATH = "data/features.npz"

In [ ]:
# import dataset

class_names = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
])
print("Jumlah kelas:", len(class_names))
print(class_names)

images, labels, paths = [], [], []

for label, cls in enumerate(class_names):
    cls_dir = os.path.join(DATA_DIR, cls)
    for fname in sorted(os.listdir(cls_dir)):
        if not fname.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        path = os.path.join(cls_dir, fname)
        img = cv2.imread(path)
        if img is None:
            print("Gagal dibaca:", path)
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        images.append(img)
        labels.append(label)
        paths.append(path)

images = np.array(images, dtype=np.uint8)
labels = np.array(labels)
print("Shape citra :", images.shape)
print("Shape label :", labels.shape)

In [ ]:
# distribusi jumlah citra per kelas

df = pd.DataFrame({"kelas": [class_names[i] for i in labels]})
counts = df["kelas"].value_counts().sort_index()

print("Total citra:", len(df))
print("Jumlah kelas:", counts.shape[0])
print("Min/Max per kelas:", counts.min(), "/", counts.max())

plt.figure(figsize=(10, 4))
counts.plot(kind="bar", color="steelblue")
plt.title("Distribusi Jumlah Citra per Kelas")
plt.xlabel("Kelas")
plt.ylabel("Jumlah citra")
plt.legend()
plt.tight_layout()
plt.savefig("results/figures/distribution_classses.png", dpi=150)
plt.show()

In [ ]:
# contoh citra dari tiap kelas

fig, axes = plt.subplots(3, 6, figsize=(14, 7))
axes = axes.ravel()

for i, cls in enumerate(class_names):
    idx = np.where(labels == i)[0][0]
    axes[i].imshow(images[idx])
    axes[i].set_title(cls, fontsize=9)
    axes[i].axis("off")

# Matikan subplot sisa yang kosong
for j in range(len(class_names), len(axes)):
    axes[j].axis("off")

plt.suptitle("Contoh Citra per Kelas", fontsize=13)
plt.tight_layout()
plt.savefig("results/figures/example_classes.png", dpi=150)
plt.show()

In [ ]:
# split data train dan test dengan stratifikasi agar proporsi tiap kelas sama di train dan test

idx_all = np.arange(len(images))

idx_train, idx_test = train_test_split(
    idx_all,
    test_size=TEST_SIZE,
    stratify=labels,
    random_state=SEED,
)

X_train_img, y_train = images[idx_train], labels[idx_train]
X_test_img,  y_test  = images[idx_test],  labels[idx_test]

print("Train:", X_train_img.shape, "| Test:", X_test_img.shape)
print("Citra per kelas di test:", np.bincount(y_test))

In [ ]:
# Augmentasi data training

def augment_image(img):
    h, w = img.shape[:2]
    out = img.copy()

    # Flip horizonta
    if random.random() < 0.5:
        out = cv2.flip(out, 1)

    # Rotasi -25 sampai 25 derajat dan zoom 1.0x sampai 1.3x
    angle = random.uniform(-25, 25)
    scale = random.uniform(1.0, 1.3)
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
    out = cv2.warpAffine(out, M, (w, h), borderMode=cv2.BORDER_REFLECT)

    # Brightness 70% sampai 130%
    factor = random.uniform(0.7, 1.3)
    out = np.clip(out.astype(np.float32) * factor, 0, 255).astype(np.uint8)

    return out

# Menerapkan augmentasi ke seluruh data training
n_asli = len(X_train_img)
aug_images, aug_labels = [], []

for img, lbl in zip(X_train_img, y_train):
    for _ in range(N_AUG):
        aug_images.append(augment_image(img))
        aug_labels.append(lbl)

X_train_img = np.concatenate([X_train_img, np.array(aug_images, dtype=np.uint8)])
y_train = np.concatenate([y_train, np.array(aug_labels)])

print("Train sebelum augmentasi :", n_asli)
print("Train setelah augmentasi :", X_train_img.shape)
print("Test (tidak diaugmentasi):", X_test_img.shape)

# Visualisasi: citra asli (kiri) dan hasil augmentasinya (kanan)
sample_kelas = [0, 5, 10, 15]
sample_idx = [np.where(y_train[:n_asli] == c)[0][0] for c in sample_kelas]

fig, axes = plt.subplots(len(sample_idx), N_AUG + 1, figsize=(3 * (N_AUG + 1), 3 * len(sample_idx)))

for r, i in enumerate(sample_idx):
    axes[r, 0].imshow(X_train_img[i])
    axes[r, 0].set_title(f"Asli ({class_names[y_train[i]]})", fontsize=9)
    axes[r, 0].axis("off")
    for k in range(N_AUG):
        axes[r, k + 1].imshow(X_train_img[n_asli + i * N_AUG + k])
        axes[r, k + 1].set_title(f"Augmentasi {k + 1}", fontsize=9)
        axes[r, k + 1].axis("off")

plt.suptitle("Contoh Hasil Augmentasi Data Train", fontsize=13)
plt.tight_layout()
plt.savefig("results/figures/example_augmentation.png", dpi=150)
plt.show()

In [ ]:
# Ekstraksi fitur citra
# Tiga fitur visual digabung jadi satu vektor:
# Warna   : histogram HSV
# Bentuk  : HOG (Histogram of Oriented Gradients)
# Tekstur : LBP (Local Binary Pattern)
def extract_features(img):
    # Fitur warna: histogram HSV 8x8x8 = 512 fitur
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    hist = cv2.calcHist([hsv], [0, 1, 2], None, [8, 8, 8],
                        [0, 180, 0, 256, 0, 256])
    hist = cv2.normalize(hist, hist).flatten()

    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

    # Fitur bentuk: HOG
    hog_feat = hog(gray, orientations=9, pixels_per_cell=(16, 16),
                   cells_per_block=(2, 2), block_norm="L2-Hys")

    # Fitur tekstur: LBP uniform (P=8, R=1) -> 10 bin
    lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")
    lbp_hist, _ = np.histogram(lbp, bins=10, range=(0, 10), density=True)

    return np.concatenate([hist, hog_feat, lbp_hist])

X_train = np.array([extract_features(im) for im in X_train_img])
X_test  = np.array([extract_features(im) for im in X_test_img])

print("Shape fitur train:", X_train.shape)
print("Shape fitur test :", X_test.shape)
print("Jumlah fitur per citra:", X_train.shape[1])

In [ ]:
# simpan fitur dan label ke file features.npz

np.savez_compressed(
    OUTPUT_PATH,
    X_train=X_train, y_train=y_train,
    X_test=X_test,   y_test=y_test,
    class_names=np.array(class_names),
)
print("Tersimpan di:", OUTPUT_PATH)